# Feature Engineering & Data Preparation

The highest-leverage work in any ML project.
Covers imputation, encoding, transforms, leakage prevention, and diagnostics.

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "figure.dpi":        130,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
    "font.family":       "sans-serif",
    "lines.linewidth":   2.2,
    "patch.edgecolor":   "none",
})
C = ["#2563EB","#DC2626","#16A34A","#D97706","#7C3AED","#0891B2","#DB2777"]
print("Setup complete")

## 1. Leakage — the most common and most damaging failure

Fitting preprocessing on the full dataset before train/test splitting
leaks test statistics into training. The accuracy looks better but the
model has peeked at the test set.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

X, y = load_breast_cancer(return_X_y=True)
cv   = StratifiedKFold(5, shuffle=True, random_state=42)

# 1. WRONG: scaler fit on full dataset
from sklearn.preprocessing import StandardScaler as SS
scores_wrong = []
for train_idx, val_idx in cv.split(X, y):
    X_sc = SS().fit_transform(X)   # fit on ALL data including val fold
    m    = LogisticRegression(max_iter=1000).fit(X_sc[train_idx], y[train_idx])
    scores_wrong.append(m.score(X_sc[val_idx], y[val_idx]))

# 2. CORRECT: scaler inside Pipeline
pipe = Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=1000))])
scores_correct = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")

# 3. Shuffle-label leakage guard
scores_shuffle = cross_val_score(
    pipe, X, np.random.default_rng(42).permutation(y),
    cv=cv, scoring="roc_auc"
)

fig, ax = plt.subplots(figsize=(10, 5))
configs = [
    ("WRONG
(scaler on full data)",  scores_wrong,   C[1]),
    ("CORRECT
(scaler in Pipeline)", scores_correct, C[2]),
    ("Shuffled labels
(leakage guard)", scores_shuffle, C[0]),
]
positions = np.arange(len(configs))
for pos, (label, scores, col) in zip(positions, configs):
    ax.scatter([pos]*len(scores), scores, color=col, alpha=0.7, s=60, zorder=3)
    ax.plot([pos-0.25, pos+0.25], [np.mean(scores)]*2, color=col, lw=3)
ax.set(xticks=positions, xticklabels=[c[0] for c in configs],
       ylabel="ROC-AUC (5-fold CV)",
       title="Leakage inflates CV score — and shuffled labels should collapse to ~0.5")
ax.axhline(0.5, color="grey", lw=1.2, linestyle="--", label="Chance (no signal)")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Leaked scaler:  {np.mean(scores_wrong):.4f} +/- {np.std(scores_wrong):.4f}")
print(f"Correct:        {np.mean(scores_correct):.4f} +/- {np.std(scores_correct):.4f}")
print(f"Shuffle guard:  {np.mean(scores_shuffle):.4f} (should be ~0.50)")

## 2. Missing data patterns and imputation strategies

Always visualise missingness before imputing. Random vs. structured missingness
demands different approaches.

In [ ]:
rng = np.random.default_rng(42)
n   = 500
df  = pd.DataFrame({
    "age":    rng.normal(45, 12, n),
    "income": rng.lognormal(10, 0.5, n),
    "score":  rng.normal(650, 80, n),
    "tenure": rng.exponential(3, n),
    "target": rng.binomial(1, 0.15, n),
})
# Introduce structured missingness: income missing more for young users
miss_prob = 0.05 + 0.3*(df["age"] < 30)
df.loc[rng.random(n) < miss_prob,      "income"] = np.nan
df.loc[rng.random(n) < 0.12,           "score"]  = np.nan
df.loc[rng.random(n) < 0.08,           "tenure"] = np.nan

from sklearn.impute import SimpleImputer

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Missing rate bar chart
ax = axes[0]
miss_rates = (df.isna().mean() * 100).sort_values(ascending=False)
miss_rates = miss_rates[miss_rates > 0]
bars = ax.bar(miss_rates.index, miss_rates.values, color=C[1], alpha=0.85)
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set(ylabel="Missing rate (%)", title="Missingness by feature")

# Missingness heatmap
ax = axes[1]
miss_matrix = df.isna().astype(int)
im = ax.imshow(miss_matrix.T, aspect="auto", cmap="Reds", interpolation="none")
ax.set(yticks=range(len(df.columns)), yticklabels=df.columns,
       xlabel="Row index", title="Missingness pattern
(red = missing)")
plt.colorbar(im, ax=ax)

# Imputation comparison: income distribution
ax = axes[2]
raw   = df["income"].dropna()
mean_imp  = df["income"].fillna(df["income"].mean())
median_imp = df["income"].fillna(df["income"].median())
ax.hist(raw.values,       bins=40, alpha=0.55, density=True, color=C[0], label=f"Observed (n={len(raw)})")
ax.hist(mean_imp.values,  bins=40, alpha=0.45, density=True, color=C[1], label="Mean imputed")
ax.hist(median_imp.values,bins=40, alpha=0.45, density=True, color=C[2], label="Median imputed")
ax.set(xlabel="Income", title="Imputation strategies for income
(mean imputation biased for log-normal)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print("Rule: mean imputation is biased for skewed distributions -> use median or model-based")

## 3. Target encoding inside cross-validation

Target encoding must be fit only on training folds to avoid leakage.
Leave-one-out or k-fold target encoding are standard safe implementations.

In [ ]:
rng = np.random.default_rng(42)
n   = 3000
cats = [f"cat_{i}" for i in range(40)]
df  = pd.DataFrame({
    "category": rng.choice(cats, n),
    "feature1": rng.normal(0, 1, n),
})
# Planted true effect: cat_0 has 3x higher conversion
df["y"] = (rng.uniform(0,1,n) < 0.05 + 0.10*(df["category"]=="cat_0")).astype(int)

from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
df["te_safe"]   = np.nan
df["te_leaked"] = df.groupby("category")["y"].transform("mean")   # WRONG: uses all data

for tr_idx, val_idx in kf.split(df):
    train_fold = df.iloc[tr_idx]
    mean_map   = train_fold.groupby("category")["y"].mean()
    global_mean = train_fold["y"].mean()
    df.loc[df.index[val_idx], "te_safe"] = (
        df.iloc[val_idx]["category"].map(mean_map).fillna(global_mean)
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in [
    (axes[0], "te_leaked", "Leaked target encoding
(fit on all data - WRONG)"),
    (axes[1], "te_safe",   "Safe target encoding
(fit per CV fold - CORRECT)"),
]:
    top = df.groupby("category")[col].mean().sort_values(ascending=False).head(10)
    color_list = [C[1] if idx=="cat_0" else C[0] for idx in top.index]
    bars = ax.bar(range(len(top)), top.values, color=color_list)
    ax.set(xticks=range(len(top)), xticklabels=top.index, ylabel="Target encoding value",
           title=title)
    ax.tick_params(axis='x', rotation=45)

fig.suptitle("cat_0 (red bar) should have highest encoding - does leakage exaggerate it?",
             fontsize=11)
plt.tight_layout()
plt.show()
leaked_cat0 = df.groupby("category")["te_leaked"].mean()["cat_0"]
safe_cat0   = df.groupby("category")["te_safe"].mean()["cat_0"]
print(f"cat_0 leaked encoding: {leaked_cat0:.4f}")
print(f"cat_0 safe encoding:   {safe_cat0:.4f}")
print(f"True conversion rate:  {df[df['category']=='cat_0']['y'].mean():.4f}")